# CO1 Java Assignment 1 — Smart-Parking System
## Google Colab executable notebook

Run **Runtime → Run all**. The notebook installs Java, executes the two “Complete the Code” exercises from the assignment, then compiles and runs an integrated Java implementation covering the main concepts and the Day-4 design challenge.

## 1. Install Java in Google Colab

In [ ]:
!apt-get -qq update
!apt-get -qq install -y openjdk-17-jdk-headless
!java -version
!javac -version

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package openjdk-17-jre-headless:amd64.
(Reading database ... 122813 files and directories currently installed.)
Preparing to unpack .../openjdk-17-jre-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Selecting previously unselected package openjdk-17-jdk-headless:amd64.
Preparing to unpack .../openjdk-17-jdk-headless_17.0.20.1+1-1~24.04_amd64.deb ...
Unpacking openjdk-17-jdk-headless:amd64 (17.0.20.1+1-1~24.04) ...
Setting up openjdk-17-jre-headless:amd64 (17.0.20.1+1-1~24.04) ...
Processing triggers for ca-certificates-java (20240118) ...
[0.031s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.031s

## 2. Task 3 — Complete the ParkingFacility code

In [ ]:
%%writefile ParkingFacilityDemo.java
import java.util.Arrays;
class ParkingFacility {
    private final String[][] slots;
    public ParkingFacility(int[] slotsPerFloor) {
        slots = new String[slotsPerFloor.length][];
        for (int f=0; f<slotsPerFloor.length; f++) slots[f]=new String[slotsPerFloor[f]];
    }
    public int[] allocate(String vehicleNo) {
        for (int f=0; f<slots.length; f++)
            for (int s=0; s<slots[f].length; s++)
                if (slots[f][s]==null) {
                    slots[f][s]=vehicleNo;
                    return new int[]{f,s};
                }
        return null;
    }
    public boolean release(int floor,int slot) {
        if(floor<0 || floor>=slots.length) return false;
        if(slot<0 || slot>=slots[floor].length) return false;
        if(slots[floor][slot]==null) return false;
        slots[floor][slot]=null;
        return true;
    }
}
public class ParkingFacilityDemo {
    public static void main(String[] args) {
        ParkingFacility p=new ParkingFacility(new int[]{2,1});
        System.out.println(Arrays.toString(p.allocate("TN01AB1111")));
        System.out.println(Arrays.toString(p.allocate("TN02CD2222")));
        System.out.println(Arrays.toString(p.allocate("TN03EF3333")));
        System.out.println(Arrays.toString(p.allocate("TN04GH4444")));
        System.out.println(p.release(0,1)+" "+p.release(0,1)+" "+p.release(5,0));
        System.out.println(Arrays.toString(p.allocate("TN04GH4444")));
    }
}

Writing ParkingFacilityDemo.java


In [ ]:
!javac ParkingFacilityDemo.java
!java ParkingFacilityDemo

## 3. Task 8 — Complete the Vehicle hierarchy and FeeDemo

In [ ]:
%%writefile FeeDemo.java
interface Chargeable { double chargingFee(int kWh); }
abstract class Vehicle {
    private final String regNo;
    protected Vehicle(String regNo){this.regNo=regNo;}
    public String getRegNo(){return regNo;}
    protected abstract double ratePerHour();
    public double calculateFee(int minutes){
        if(minutes<0) throw new IllegalArgumentException("Negative minutes");
        if(minutes==0) return 0;
        int hours=(minutes+59)/60;
        return hours*ratePerHour();
    }
    @Override public boolean equals(Object o){
        return o instanceof Vehicle && regNo.equals(((Vehicle)o).regNo);
    }
    @Override public int hashCode(){return regNo.hashCode();}
}
class TwoWheeler extends Vehicle{
    TwoWheeler(String r){super(r);}
    @Override protected double ratePerHour(){return 10;}
}
class Car extends Vehicle{
    Car(String r){super(r);}
    @Override protected double ratePerHour(){return 30;}
}
class ElectricCar extends Car implements Chargeable{
    ElectricCar(String r){super(r);}
    @Override public double chargingFee(int kWh){
        if(kWh<0) throw new IllegalArgumentException("Negative kWh");
        return kWh*12.0;
    }
}
public class FeeDemo{
    public static void main(String[] args){
        Vehicle[] parked={new TwoWheeler("TN01A1"),new Car("TN02B2"),new ElectricCar("TN03C3")};
        for(Vehicle v:parked){
            double fee=v.calculateFee(130);
            if(v instanceof Chargeable c) fee+=c.chargingFee(10);
            System.out.println(v.getClass().getSimpleName()+" "+v.getRegNo()+" -> Rs "+fee);
        }
        System.out.println(new Car("TN02B2").equals(new ElectricCar("TN02B2")));
    }
}

Writing FeeDemo.java


In [ ]:
!javac FeeDemo.java
!java FeeDemo

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
TwoWheeler TN01A1 -> Rs 30.0
Car TN02B2 -> Rs 90.0
ElectricCar TN03C3 -> Rs 210.0
true


## 4. Tasks 1–12 — Integrated Java implementation
The following source includes: vehicle hierarchy, abstract class, interfaces, monthly and one-day passes, ticket counter, jagged floor/slot array, allocation/release, active ticket map, fee calculation, e-bike and delivery van categories.

In [ ]:
%%writefile SmartParkingSystem.java
import java.time.*;
import java.util.*;

interface Chargeable { double chargingFee(int kWh); }

interface Pass {
    boolean isValidFor(LocalDateTime entry, LocalDateTime exit);
}

final class MonthlyPass implements Pass {
    private final LocalDateTime from, until;
    MonthlyPass(LocalDateTime from, LocalDateTime until) {
        if (until.isBefore(from)) throw new IllegalArgumentException("Invalid pass dates");
        this.from = from; this.until = until;
    }
    public boolean isValidFor(LocalDateTime in, LocalDateTime out) {
        return !in.isBefore(from) && !out.isAfter(until);
    }
}

final class OneDayPass implements Pass {
    private final LocalDate date;
    OneDayPass(LocalDate date) { this.date = date; }
    public boolean isValidFor(LocalDateTime in, LocalDateTime out) {
        return in.toLocalDate().equals(date) && out.toLocalDate().equals(date);
    }
}

abstract class Vehicle {
    private final String regNo;
    private final Pass pass;
    protected Vehicle(String regNo) { this(regNo, null); }
    protected Vehicle(String regNo, Pass pass) {
        if (regNo == null || regNo.isBlank()) throw new IllegalArgumentException("Registration required");
        this.regNo = regNo.trim().toUpperCase(Locale.ROOT);
        this.pass = pass;
    }
    public final String getRegNo() { return regNo; }
    public final Pass getPass() { return pass; }
    protected abstract double ratePerHour();

    public double calculateFee(int minutes) {
        if (minutes < 0) throw new IllegalArgumentException("Negative duration");
        if (minutes == 0) return 0;
        int hours = (minutes + 59) / 60;
        return hours * ratePerHour();
    }
    public double calculateFee(LocalDateTime in, LocalDateTime out) {
        if (out.isBefore(in)) throw new IllegalArgumentException("Exit before entry");
        return calculateFee(Math.toIntExact(Duration.between(in, out).toMinutes()));
    }
    @Override public boolean equals(Object obj) {
        return obj instanceof Vehicle && regNo.equals(((Vehicle)obj).regNo);
    }
    @Override public int hashCode() { return regNo.hashCode(); }
    @Override public String toString() { return getClass().getSimpleName() + "[" + regNo + "]"; }
}

class TwoWheeler extends Vehicle {
    TwoWheeler(String r) { super(r); }
    TwoWheeler(String r, Pass p) { super(r,p); }
    protected double ratePerHour() { return 10; }
}
class Car extends Vehicle {
    Car(String r) { super(r); }
    Car(String r, Pass p) { super(r,p); }
    protected double ratePerHour() { return 30; }
}
class ElectricCar extends Car implements Chargeable {
    ElectricCar(String r) { super(r); }
    ElectricCar(String r, Pass p) { super(r,p); }
    public double chargingFee(int kWh) {
        if (kWh < 0) throw new IllegalArgumentException("Negative kWh");
        return 12.0 * kWh;
    }
}
class EBike extends TwoWheeler implements Chargeable {
    EBike(String r) { super(r); }
    public double chargingFee(int kWh) {
        if (kWh < 0) throw new IllegalArgumentException("Negative kWh");
        return 20.0; // flat fee per charging session
    }
}
class DeliveryVan extends Vehicle {
    DeliveryVan(String r) { super(r); }
    protected double ratePerHour() { return 60; } // twice the car rate
    @Override public double calculateFee(LocalDateTime in, LocalDateTime out) {
        if (out.isBefore(in)) throw new IllegalArgumentException("Exit before entry");
        long mins = Duration.between(in, out).toMinutes();
        return calculateFee(Math.toIntExact(mins));
    }
}

final class ParkingTicket {
    private static long nextId = 1000;
    private final long ticketNo;
    private final Vehicle vehicle;
    private final int floor, slot;
    private final LocalDateTime entryTime;
    ParkingTicket(Vehicle v, int f, int s, LocalDateTime in) {
        ticketNo = nextId++;
        vehicle = Objects.requireNonNull(v);
        if (f < 0 || s < 0) throw new IllegalArgumentException("Invalid position");
        floor=f; slot=s; entryTime=Objects.requireNonNull(in);
    }
    public long getTicketNo() { return ticketNo; }
    public Vehicle getVehicle() { return vehicle; }
    public int getFloor() { return floor; }
    public int getSlot() { return slot; }
    public LocalDateTime getEntryTime() { return entryTime; }
}

final class ParkingFacility {
    private final String[][] slots; // jagged array: different slots per floor
    private final Map<String, ParkingTicket> active = new HashMap<>();

    ParkingFacility(int[] slotsPerFloor) {
        if (slotsPerFloor == null || slotsPerFloor.length == 0)
            throw new IllegalArgumentException("At least one floor required");
        slots = new String[slotsPerFloor.length][];
        for (int f=0; f<slotsPerFloor.length; f++) {
            if (slotsPerFloor[f] < 1) throw new IllegalArgumentException("Invalid floor capacity");
            slots[f] = new String[slotsPerFloor[f]];
        }
    }

    public int[] allocate(String reg) {
        for (int f=0; f<slots.length; f++)
            for (int s=0; s<slots[f].length; s++)
                if (slots[f][s] == null) {
                    slots[f][s] = reg;
                    return new int[]{f,s};
                }
        return null;
    }

    public boolean release(int f, int s) {
        if (f<0 || f>=slots.length || s<0 || s>=slots[f].length) return false;
        if (slots[f][s] == null) return false;
        slots[f][s] = null;
        return true;
    }

    public ParkingTicket park(Vehicle v, LocalDateTime entry) {
        String reg = v.getRegNo();
        if (active.containsKey(reg)) throw new IllegalArgumentException("Duplicate active registration");
        int[] pos = allocate(reg);
        if (pos == null) throw new IllegalStateException("Parking full");
        ParkingTicket ticket = new ParkingTicket(v,pos[0],pos[1],entry);
        active.put(reg,ticket);
        return ticket;
    }

    public double exit(String reg, LocalDateTime out, int kWh) {
        String key=reg.trim().toUpperCase(Locale.ROOT);
        ParkingTicket t=active.get(key);
        if (t==null) throw new IllegalArgumentException("No active ticket");
        Vehicle v=t.getVehicle();
        double fee;
        if (v.getPass()!=null && v.getPass().isValidFor(t.getEntryTime(),out)) fee=0;
        else {
            fee=v.calculateFee(t.getEntryTime(),out);
            if (v instanceof Chargeable c) fee += c.chargingFee(kWh);
        }
        release(t.getFloor(),t.getSlot());
        active.remove(key);
        return fee;
    }
    public int activeCount() { return active.size(); }
}

public class SmartParkingSystem {
    public static void main(String[] args) {
        System.out.println("TASK 3 - allocation/release");
        ParkingFacility p = new ParkingFacility(new int[]{2,1});
        System.out.println(Arrays.toString(p.allocate("TN01AB1111")));
        System.out.println(Arrays.toString(p.allocate("TN02CD2222")));
        System.out.println(Arrays.toString(p.allocate("TN03EF3333")));
        System.out.println(Arrays.toString(p.allocate("TN04GH4444")));
        System.out.println(p.release(0,1)+" "+p.release(0,1)+" "+p.release(5,0));
        System.out.println(Arrays.toString(p.allocate("TN04GH4444")));

        System.out.println("\\nTASK 8 - polymorphism and fee demo");
        Vehicle[] vehicles={new TwoWheeler("TN01A1"),new Car("TN02B2"),new ElectricCar("TN03C3")};
        for (Vehicle v:vehicles) {
            double fee=v.calculateFee(130);
            if(v instanceof Chargeable c) fee+=c.chargingFee(10);
            System.out.println(v.getClass().getSimpleName()+" "+v.getRegNo()+" -> Rs "+fee);
        }
        System.out.println(new Car("TN02B2").equals(new ElectricCar("TN02B2")));

        System.out.println("\\nTASK 12 - new categories");
        EBike eb=new EBike("TN04E4");
        DeliveryVan van=new DeliveryVan("TN05V5");
        System.out.println("EBike flat charging fee -> Rs "+eb.chargingFee(5));
        System.out.println("DeliveryVan 60 minutes -> Rs "+van.calculateFee(60));

        System.out.println("\\nINTEGRATED FACILITY TEST");
        ParkingFacility facility=new ParkingFacility(new int[]{1,3,2,4,1});
        LocalDateTime now=LocalDateTime.of(2026,10,31,10,0);
        ParkingTicket ticket=facility.park(new Car("TN10XY1000"),now);
        System.out.println("Ticket number: "+ticket.getTicketNo());
        System.out.println("Exit fee for 61 minutes: Rs "+
            facility.exit("TN10XY1000",now.plusMinutes(61),0));
        System.out.println("Active vehicles after exit: "+facility.activeCount());
    }
}


Writing SmartParkingSystem.java


## 5. Compile integrated Java source

In [ ]:
!javac SmartParkingSystem.java
print("Compilation successful.")

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
Compilation successful.


## 6. Run integrated Java source

In [ ]:
!java SmartParkingSystem

[0.001s][warning][os,container] Cgroup memory controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
[0.001s][warning][os,container] Cgroup cpu controller path at '/sys/fs/cgroup' seems to have moved to '/../../jupyter-children', detected limits won't be accurate
TASK 3 - allocation/release
[0, 0]
[0, 1]
[1, 0]
null
true false false
[0, 1]
\nTASK 8 - polymorphism and fee demo
TwoWheeler TN01A1 -> Rs 30.0
Car TN02B2 -> Rs 90.0
ElectricCar TN03C3 -> Rs 210.0
true
\nTASK 12 - new categories
EBike flat charging fee -> Rs 20.0
DeliveryVan 60 minutes -> Rs 60.0
\nINTEGRATED FACILITY TEST
Ticket number: 1000
Exit fee for 61 minutes: Rs 60.0
Active vehicles after exit: 0


## 7. Task mapping and submission

| Task | Code / evidence |
|---|---|
| 1. Objects and responsibilities | Vehicle, ParkingTicket, ParkingFacility, Pass |
| 2. Data representation | Jagged `String[][]`, `int`, `long`, `LocalDateTime` |
| 3. Control logic | `ParkingFacilityDemo.java` |
| 4. Classes and constructors | Vehicle hierarchy and ticket classes |
| 5. Static members | `ParkingTicket.nextId` |
| 6. Inheritance | TwoWheeler, Car, ElectricCar, EBike, DeliveryVan |
| 7. Overriding | `ratePerHour()`, `equals()`, `hashCode()` |
| 8. Polymorphism | `FeeDemo.java`, `Vehicle[]` |
| 9. Abstraction | Abstract Vehicle, Chargeable and Pass interfaces |
| 10. Encapsulation | Private fields and validation |
| 11. Implementation/testing | Integrated facility test |
| 12. Design challenge | EBike, DeliveryVan, OneDayPass, jagged array |

**Submission steps:** Run all cells in Colab, save the notebook with outputs, download the `.ipynb`, and upload it with your Java source files and your own screenshots to the required GitHub repository. Use the accompanying DOCX for the written analysis, pseudocode, UML-style diagrams and complexity discussion.